In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
import json
import re
from torch.utils.data import Dataset, DataLoader

class MessageDataset(Dataset):
    def __init__(self, json_files, tokenizer=None, sequence_length=512):
        self.json_files = json_files
        self.file_cache = {}  # Cache loaded files
        self.sequences = []  # Store conversation sequences
        self.speaker_to_id = {}  # Maps speaker names to speaker IDs
        self.tokenizer = tokenizer  # Optional tokenizer
        self.sequence_length = sequence_length
        
        # First pass: collect all unique speakers
        all_speakers = set()
        for file_path in json_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                messages = data.get('messages', [])
                for msg in messages:
                    if ('content' in msg and 
                        not self.is_reaction(msg.get('content', '')) and 
                        not self.contains_emoji(msg.get('content', ''))):
                        all_speakers.add(msg['sender_name'])
        
        # Assign speaker IDs
        for idx, speaker in enumerate(sorted(all_speakers), start=1):
            self.speaker_to_id[speaker] = f"<S{idx}>"
        
        # Second pass: build conversation sequences
        for file_path in json_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                messages = data.get('messages', [])
                
                # Filter valid messages
                valid_messages = []
                for msg in messages:
                    if ('content' in msg and 
                        not self.is_reaction(msg.get('content', '')) and 
                        not self.contains_emoji(msg.get('content', ''))):
                        valid_messages.append(msg)
                
                # Create sequences by concatenating messages up to sequence_length
                self._create_sequences(valid_messages)
    
    def _create_sequences(self, messages):
        """Create conversation sequences from messages"""
        current_sequence = []
        current_length = 0
        
        for msg in messages:
            speaker_token = self.speaker_to_id[msg['sender_name']]
            cleaned_text = self.clean_text(msg['content'])
            formatted_msg = f"{speaker_token} {cleaned_text} <EOM>"
            
            # Estimate token length (rough approximation)
            if self.tokenizer:
                msg_tokens = self.tokenizer.encode(formatted_msg)
                msg_length = len(msg_tokens)
            else:
                msg_length = len(formatted_msg.split())
            
            # If adding this message would exceed sequence_length, save current sequence and start new one
            if current_length + msg_length > self.sequence_length and current_sequence:
                self.sequences.append(''.join(current_sequence))
                current_sequence = []
                current_length = 0
            
            current_sequence.append(formatted_msg)
            current_length += msg_length
        
        # Add the last sequence if it exists
        if current_sequence:
            self.sequences.append(''.join(current_sequence))
    
    def is_reaction(self, text):
        """Check if message is a reaction"""
        return 'reacted' in text.lower() and 'to your message' in text.lower()
    
    def contains_emoji(self, text):
        """Check if text contains emojis"""
        # Emoji unicode ranges
        emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"  # emoticons
            "\U0001F300-\U0001F5FF"  # symbols & pictographs
            "\U0001F680-\U0001F6FF"  # transport & map symbols
            "\U0001F1E0-\U0001F1FF"  # flags (iOS)
            "\U00002702-\U000027B0"  # dingbats
            "\U000024C2-\U0001F251"
            "\U0001F900-\U0001F9FF"  # supplemental symbols
            "\U0001FA00-\U0001FA6F"  # chess symbols
            "\U00002600-\U000026FF"  # misc symbols
            "\U00002B50"              # star
            "]+", 
            flags=re.UNICODE
        )
        return bool(emoji_pattern.search(text))
    
    def clean_text(self, text):
        # Encode back to latin-1 and decode as utf-8 to fix mojibake
        try:
            text = text.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass  # Keep original if conversion fails
        
        text = ' '.join(text.split())
        text = re.sub(r'http\S+|www\S+', '', text)
        return text.strip()
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        text = self.sequences[idx]
        
        # If tokenizer is provided, tokenize
        if self.tokenizer:
            encoded = self.tokenizer.encode(
                text,
                max_length=self.sequence_length,
                truncation=True
            )
            return torch.tensor(encoded, dtype=torch.long)
        
        # Otherwise return text
        return text

In [5]:
# Collate function to pad batches (used by DataLoader)
def collate_fn(batch):
    """Pads sequences to the same length within a batch"""
    max_len = max(len(seq) for seq in batch)
    # Use the tokenizer's pad_token_id instead of 0
    padded = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    
    for i, seq in enumerate(batch):
        padded[i, :len(seq)] = seq
    
    return padded

In [6]:
# Install transformers if needed
# !pip install transformers

from transformers import GPT2Tokenizer

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add special tokens for speaker IDs and end of message
special_tokens = {
    'additional_special_tokens': ['<S1>', '<S2>', '<S3>', '<S4>', '<S5>', '<S6>', '<S7>', '<S8>', '<EOM>']
}
num_added = tokenizer.add_special_tokens(special_tokens)

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS token as padding token

sequence_length=512
vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")
print(f"Number of special tokens added: {num_added}")
print(f"Padding token ID: {tokenizer.pad_token_id}")

/opt/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocabulary size: 50266
Number of special tokens added: 9
Padding token ID: 50256


In [10]:
# Define file paths
json_files = [
    'data/message_1.json',
    'data/message_2.json',
    'data/message_3.json',
    'data/message_4.json',
    'data/message_5.json',
    'data/chris_message_1.json'
]

In [13]:
# Create tokenized dataset
dataset = MessageDataset(json_files, tokenizer=tokenizer)

# Create DataLoader with custom collate function
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

# Test
for batch in dataloader:
    x = batch
    print(f"\nBatch shape: {batch.shape}")
    print(f"First sequence tokens: {batch[0][:20]}")
    print(f"Decoded: {tokenizer.decode(batch[0][:20])}")
    break

Dataset size: 829
Number of batches: 26

Batch shape: torch.Size([32, 512])
First sequence tokens: tensor([50257,   327, 29480,  9891,  1034,   220, 50265, 50257,   679,   373,
         5762,   257,  9335,   286,    66,   220, 50265, 50261,  4889,   683])
Decoded: <S1>  Cottage cheese op  <EOM> <S1>  He was wearing a mask ofc  <EOM> <S5>  Call him


In [14]:
# Create input/target pairs for GPT training
# Input: all tokens except the last one
# Target: all tokens except the first one (shifted by 1)

def create_input_target_pairs(batch):
    """
    Creates input and target sequences for next-token prediction.
    
    Args:
        batch: Tensor of shape [batch_size, seq_len]
    
    Returns:
        inputs: Tensor of shape [batch_size, seq_len-1]
        targets: Tensor of shape [batch_size, seq_len-1]
    """
    inputs = batch[:, :-1]   # All tokens except the last
    targets = batch[:, 1:]   # All tokens except the first
    return inputs, targets

# Example usage
for batch in dataloader:
    inputs, targets = create_input_target_pairs(batch)
    
    print(f"Original batch shape: {batch.shape}")
    print(f"Input shape: {inputs.shape}")
    print(f"Target shape: {targets.shape}")
    print(f"\nExample from first sequence:")
    print(f"Input tokens (first 10): {inputs[0][:10]}")
    print(f"Target tokens (first 10): {targets[0][:10]}")
    print(f"\nInput decoded: {tokenizer.decode(inputs[0][:20])}")
    print(f"Target decoded: {tokenizer.decode(targets[0][:20])}")
    break

Original batch shape: torch.Size([32, 512])
Input shape: torch.Size([32, 511])
Target shape: torch.Size([32, 511])

Example from first sequence:
Input tokens (first 10): tensor([50261,  2488, 48526,  9552,   373,  4019,     4,   286,   534,  1074])
Target tokens (first 10): tensor([ 2488, 48526,  9552,   373,  4019,     4,   286,   534,  1074,  8104])

Input decoded: <S5>  @Meta AI was 80% of your team laid off because of these shitty responses?  <EOM>
Target decoded:  @Meta AI was 80% of your team laid off because of these shitty responses?  <EOM> <S5>


# Model Architecture

1. Embedding layer: convert tokens into embeddings
2. Positional embeddings: add position information (sinusodal)
3. Transformer blocks
    - layer norm
    - multi-head attention
    - res connection
    - layer norm
    - feed forward (linear)
    - res connection
4. Output head

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # Project to Q, K, V
        Q = self.W_q(x)  # (B, T, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape into (B, n_heads, T, head_dim)
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, n_heads, T, T)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values
        out = attn_weights @ V  # (B, n_heads, T, head_dim)

        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)
        return out

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        # Pre-norm: LayerNorm -> MHA -> Residual
        x = x + self.attn(self.ln1(x), mask)
        # Pre-norm: LayerNorm -> FFN -> Residual
        x = x + self.ff(self.ln2(x))
        return x


class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024,
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Token + positional embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # Final layer norm + output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share token embedding weights with output head
        self.head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device

        # Embeddings
        tok_emb = self.token_emb(idx)                          # (B, T, d_model)
        pos_emb = self.pos_emb(torch.arange(T, device=device)) # (T, d_model)
        x = self.drop(tok_emb + pos_emb)

        # Causal mask: prevent attending to future tokens
        mask = torch.tril(torch.ones(T, T, device=device)).unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, mask)

        # Output
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                                   ignore_index=tokenizer.pad_token_id)
        return logits, loss

In [11]:
# Instantiate model
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

model = Transformer(
    vocab_size=vocab_size,
    d_model=256,
    n_heads=8,
    n_layers=6,
    d_ff=1024,
    max_seq_len=sequence_length,
    dropout=0.1,
).to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Sanity check with one batch
for batch in dataloader:
    inputs, targets = create_input_target_pairs(batch)
    inputs, targets = inputs.to(device), targets.to(device)
    logits, loss = model(inputs, targets)
    print(f"Input shape:  {inputs.shape}")
    print(f"Logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")
    break

Using device: mps
Total parameters: 17,738,240
Input shape:  torch.Size([32, 511])
Logits shape: torch.Size([32, 511, 50266])
Loss: 10.9023


In [12]:
# Count actual tokens (before padding)
total_tokens = sum(len(tokenizer.encode(seq)) for seq in dataset.sequences)
print(f"Total tokens: {total_tokens:,}")
print(f"Total sequences: {len(dataset.sequences)}")
print(f"Avg tokens per sequence: {total_tokens / len(dataset.sequences):.0f}")

Total tokens: 415,814
Total sequences: 829
Avg tokens per sequence: 502


In [13]:
# Training loop
epochs = 50
lr = 3e-4

optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

model.train()
for epoch in range(epochs):
    total_loss = 0
    num_batches = 0

    for batch in dataloader:
        inputs, targets = create_input_target_pairs(batch)
        inputs, targets = inputs.to(device), targets.to(device)

        logits, loss = model(inputs, targets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient Clipping
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    scheduler.step()
    avg_loss = total_loss / num_batches

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

KeyboardInterrupt: 